In [4]:
%%writefile app1.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# ----------------------------
# Streamlit Config
# ----------------------------
st.set_page_config(page_title="CO₂ Emisyon Dashboard – OWID", layout="wide")

DATA_PATH = "owid-co2-data.csv"

@st.cache_data
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Gerçek ülkeler
    df = df[df["iso_code"].notna()].copy()

    cols = [
        "country", "iso_code", "year",
        "co2", "co2_per_capita",
        "population", "gdp",
        "total_ghg", "total_ghg_excluding_lucf"
    ]
    existing_cols = [c for c in cols if c in df.columns]

    df = df[existing_cols]
    df = df.sort_values(["country", "year"])
    return df

df = load_data(DATA_PATH)

countries = sorted(df["country"].unique())
min_year = int(df["year"].min())
max_year = int(df["year"].max())

# ----------------------------
# Helpers
# ----------------------------
def get_country_series(country: str) -> pd.DataFrame:
    cdf = df[df["country"] == country].copy()
    cdf = cdf.sort_values("year")
    return cdf[cdf["co2_per_capita"].notna()]

def fit_model(X, y, poly_degree=2):
    """Polynomial Regression veya Linear Regression kurar."""
    X = np.asarray(X).reshape(-1, 1)
    y = np.asarray(y)

    if poly_degree is None:
        model = LinearRegression()
        model.fit(X, y)
        return model, None

    poly = PolynomialFeatures(degree=poly_degree)
    X_poly = poly.fit_transform(X)
    model = LinearRegression()
    model.fit(X_poly, y)

    return model, poly

def predict(model, years, transformer=None):
    years = np.asarray(years).reshape(-1, 1)
    if transformer is not None:
        X = transformer.transform(years)
    else:
        X = years
    return model.predict(X)

# ----------------------------
# Sidebar Menu
# ----------------------------
menu = st.sidebar.selectbox(
    "📌 Sayfa Seç",
    [
        "Ana Sayfa",
        "Ülke Analizi",
        "Top 10 Artış (2000–2024)",
        "Top 10 Azalış (2000–2024)",
        "Dağılım Grafiği (2000 vs 2024)",
        "Tahmin Grafiği (Tek Ülke)",
        "2050 Tahmini – Ülkeler Karşılaştırma"
    ]
)

# ----------------------------
# 1) Ana Sayfa
# ----------------------------
if menu == "Ana Sayfa":
    st.title("🌍 Küresel CO₂ Emisyon Dashboard – OWID Verisi")

    latest_year = max_year
    latest = df[(df["year"] == latest_year) & (df["co2_per_capita"].notna())]

    col1, col2, col3 = st.columns(3)
    col1.metric("Ülke Sayısı", latest["country"].nunique())
    col2.metric("Ort. Kişi Başı CO₂", round(latest["co2_per_capita"].mean(), 3))
    col3.metric("Toplam CO₂", round(latest["co2"].sum(), 2) if "co2" in latest else "Yok")

    st.subheader("⬆️ En Yüksek 10 Ülke (Kişi başı CO₂)")
    st.dataframe(latest.sort_values("co2_per_capita", ascending=False).head(10)[["country","co2_per_capita"]])

    st.subheader("⬇️ En Düşük 10 Ülke (Kişi başı CO₂)")
    st.dataframe(latest.sort_values("co2_per_capita", ascending=True).head(10)[["country","co2_per_capita"]])

# ----------------------------
# 2) Ülke Analizi
# ----------------------------
elif menu == "Ülke Analizi":
    st.title("🏳️ Ülke Bazlı CO₂ Analizi")

    selected_countries = st.multiselect("Ülke seç:", countries, default=["Turkey"])

    fig, ax = plt.subplots(figsize=(10, 5))

    for c in selected_countries:
        cdf = get_country_series(c)
        ax.plot(cdf["year"], cdf["co2_per_capita"], marker="o", label=c)

    ax.set_xlabel("Yıl")
    ax.set_ylabel("Kişi Başı CO₂")
    ax.legend()
    st.pyplot(fig)

# ----------------------------
# 3) Top 10 Artış
# ----------------------------
elif menu == "Top 10 Artış (2000–2024)":
    st.title("📈 2000–2024 Arası En Fazla Artış Gösteren Ülkeler")

    subset = df[df["year"].isin([2000, 2024])]
    pivot = subset.pivot(index="country", columns="year", values="co2_per_capita").dropna()

    pivot["change"] = pivot[2024] - pivot[2000]
    pivot["pct"] = pivot["change"] / pivot[2000] * 100

    st.dataframe(pivot.sort_values("pct", ascending=False).head(10))

# ----------------------------
# 4) Top 10 Azalış
# ----------------------------
elif menu == "Top 10 Azalış (2000–2024)":
    st.title("📉 2000–2024 Arası En Fazla Azalan 10 Ülke")

    subset = df[df["year"].isin([2000, 2024])]
    pivot = subset.pivot(index="country", columns="year", values="co2_per_capita").dropna()

    pivot["change"] = pivot[2024] - pivot[2000]
    pivot["pct"] = pivot["change"] / pivot[2000] * 100

    st.dataframe(pivot.sort_values("pct").head(10))

# ----------------------------
# 5) Scatter Plot
# ----------------------------
elif menu == "Dağılım Grafiği (2000 vs 2024)":
    st.title("🟣 2000 vs 2024 CO₂ Scatter")

    subset = df[df["year"].isin([2000, 2024])]
    pivot = subset.pivot(index="country", columns="year", values="co2_per_capita").dropna()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(pivot[2000], pivot[2024])
    ax.set_xlabel("2000")
    ax.set_ylabel("2024")
    ax.set_title("2000 vs 2024 Scatter")
    st.pyplot(fig)

# ----------------------------
# 6) Tek Ülke Tahmini
# ----------------------------
elif menu == "Tahmin Grafiği (Tek Ülke)":
    st.title("🔮 Tek Ülke 2050 Tahmini")

    country = st.selectbox("Ülke seç:", countries, index=countries.index("Turkey") if "Turkey" in countries else 0)

    cdf = get_country_series(country)
    if cdf.shape[0] < 5:
        st.warning("Bu ülke için yeterli veri yok.")
    else:
        X = cdf["year"].values
        y = cdf["co2_per_capita"].values

        # Polynomial model
        model, transformer = fit_model(X, y, poly_degree=2)

        future_years = np.arange(int(X.min()), 2051)
        preds = predict(model, future_years, transformer)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.scatter(X, y, color="red", label="Gerçek Veri")
        ax.plot(future_years, preds, label="Polynomial Tahmin (2050)")

        ax.legend()
        st.pyplot(fig)

# ----------------------------
# 7) Tüm Ülkeler için 2050 Tahmini
# ----------------------------
elif menu == "2050 Tahmini – Ülkeler Karşılaştırma":
    st.title("🌐 Tüm Ülkeler için 2050 CO₂ Tahmini")

    results = []
    for c in countries:
        cdf = get_country_series(c)
        if cdf.shape[0] < 8:
            continue

        X = cdf["year"].values
        y = cdf["co2_per_capita"].values

        model, transformer = fit_model(X, y, poly_degree=2)
        pred_2050 = predict(model, [2050], transformer)[0]

        results.append([c, pred_2050])

    res_df = pd.DataFrame(results , columns=["country", "co2_2050"])
    st.dataframe(res_df.sort_values("co2_2050", ascending=False).head(20))


Overwriting app1.py
